# Download and refresh local price CSV files

Uses `download_adj_close_prices` from `data_loader.py` with provider fallback chain: `yfinance` -> `defeatbeta_api`.

It downloads each ticker in a specified date range and saves one CSV per ticker in `data/` with the same format used by the project (`Date,Adj Close`).

In [22]:
from __future__ import annotations

from pathlib import Path
import importlib
import time
import pandas as pd

import data_loader as _data_loader
importlib.reload(_data_loader)
download_adj_close_prices = _data_loader.download_adj_close_prices

In [23]:
TICKERS = [
    "CSSPX.MI",
    "AVUV",
    "VTI",
    "VBR",
    "IEF",
    "VUTY.AS",
    "DBC", # Commodities
    "GSG", # Commodities
    "SWDA.MI",
    "EIMI.MI",
    "CSNDX.MI",
    "IUSQ.MI",
    "VWCE.MI",
    "EXS1.MI",
    "MEUD.MI",
    "SMEA.MI",
    "XD9U.MI",
    "XMME.MI",
    "CSSX5E.MI",
    "EMUL.MI",
    "XLRE",
    "GOVT",
    "TLH",
    "LTPZ",
    "XTLT.TO",
    "UTHY",
    "TIP",
    "GLD",
    "IGLA.L",
    "XG7S.DE",
    "SEGA.MI",
    "VAGF.DE",
    "BTC=F",
    "SI=F",
    "CL=F",
    "CSH.PA",
    "PDBC",
    "VGLT",
    "VGSH",
    "VWRA.L",
]

In [24]:
# Simple updater focused on VTI and VBR only

# Used only if a local file does not exist yet
FALLBACK_START_DATE = "2010-04-03"
END_DATE = "2020-04-20"

# Small delay between requests helps reduce provider throttling
COOLDOWN_SECONDS = 4

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tickers: {TICKERS}")
print(f"Fallback start: {FALLBACK_START_DATE}")
print(f"End date: {END_DATE}")
print(f"Output dir: {DATA_DIR.resolve()}")

Tickers: ['CSSPX.MI', 'AVUV', 'VTI', 'VBR', 'IEF', 'VUTY.AS', 'DBC', 'GSG', 'SWDA.MI', 'EIMI.MI', 'CSNDX.MI', 'IUSQ.MI', 'VWCE.MI', 'EXS1.MI', 'MEUD.MI', 'SMEA.MI', 'XD9U.MI', 'XMME.MI', 'CSSX5E.MI', 'EMUL.MI', 'XLRE', 'GOVT', 'TLH', 'LTPZ', 'XTLT.TO', 'UTHY', 'TIP', 'GLD', 'IGLA.L', 'XG7S.DE', 'SEGA.MI', 'VAGF.DE', 'BTC=F', 'SI=F', 'CL=F', 'CSH.PA', 'PDBC', 'VGLT', 'VGSH', 'VWRA.L']
Fallback start: 2010-04-03
End date: 2020-04-20
Output dir: /Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data


In [25]:
results = []

for i, ticker in enumerate(TICKERS, start=1):
    print(f"[{i}/{len(TICKERS)}] Updating {ticker}...")
    status = "ok"
    rows = 0
    downloaded_rows = 0
    error = ""

    out_path = DATA_DIR / f"{ticker}.csv"
    req_end = pd.Timestamp(END_DATE)
    one_day = pd.Timedelta(days=1)

    try:
        existing_series = None
        start_ts = pd.Timestamp(FALLBACK_START_DATE)

        if out_path.exists():
            existing_df = pd.read_csv(out_path, parse_dates=["Date"])
            if not existing_df.empty and {"Date", "Adj Close"}.issubset(existing_df.columns):
                existing_series = (
                    existing_df.set_index("Date")["Adj Close"]
                    .astype(float)
                    .dropna()
                    .sort_index()
                )
                if not existing_series.empty:
                    start_ts = existing_series.index.max() + one_day

        if start_ts > req_end:
            print("    already up to date; no download needed")
            final_series = existing_series
            if final_series is None or final_series.empty:
                raise RuntimeError("No local data available and no date range to download.")
        else:
            new_prices = download_adj_close_prices(
                tickers=[ticker],
                start=start_ts.strftime("%Y-%m-%d"),
                end=req_end.strftime("%Y-%m-%d"),
            )
            new_series = new_prices[ticker].dropna().sort_index()
            downloaded_rows = int(new_series.shape[0])

            if existing_series is not None and not existing_series.empty:
                final_series = pd.concat([existing_series, new_series]).sort_index()
                final_series = final_series[~final_series.index.duplicated(keep="last")]
            else:
                final_series = new_series

        out_df = final_series.to_frame(name="Adj Close")
        out_df.index.name = "Date"
        out_df.to_csv(out_path, date_format="%Y-%m-%d")

        rows = int(out_df.shape[0])
        print(f"    downloaded {downloaded_rows} new rows; total rows now {rows}")
    except Exception as exc:
        status = "failed"
        error = str(exc)
        print(f"    FAILED: {ticker}: {error}")

    results.append(
        {
            "ticker": ticker,
            "status": status,
            "downloaded_rows": downloaded_rows,
            "total_rows": rows,
            "error": error,
        }
    )

    if i < len(TICKERS):
        time.sleep(COOLDOWN_SECONDS)

summary = pd.DataFrame(results)
summary

[1/40] Updating CSSPX.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4038
[2/40] Updating AVUV...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 1645
[3/40] Updating VTI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4036
[4/40] Updating VBR...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4024
[5/40] Updating IEF...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4032
[6/40] Updating VUTY.AS...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 2592
[7/40] Updating DBC...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4036
[8/40] Updating GSG...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4036
[9/40] Updating SWDA.MI...
    already up to date; no download needed
    downloaded 0

/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:324: RuntimeWarning: Local cache miss for symbols: ['IUSQ.MI']. Attempting online providers.
  warnings.warn(

1 Failed download:
['IUSQ.MI']: YFTzMissingError('possibly delisted; no timezone found')


[12/40] Updating IUSQ.MI...


/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:377: RuntimeWarning: Batch yfinance download returned incomplete data. Falling back to per-ticker recovery for: ['IUSQ.MI']
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:386: RuntimeWarning: Yahoo Finance batch response returned no usable rows for all uncached symbols. Likely temporary provider throttling; retrying per ticker with backoff.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:403: RuntimeWarning: Waiting 7.7s before per-ticker Yahoo retry for IUSQ.MI after batch provider failure.
  warnings.warn(

1 Failed download:
['IUSQ.MI']: YFTzMissingError('possibly delisted; no timezone found')
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data_loader.py:179: RuntimeWarning: defeatbeta_api is not installed/importable, so fallback data cannot be fetched.
  warnings.warn(
/Users/giacomomaggiore/Desktop/CODING/leveraged-etf-pf/data

    FAILED: IUSQ.MI: No price data returned by yfinance for requested inputs.
[13/40] Updating VWCE.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 1587
[14/40] Updating EXS1.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4594
[15/40] Updating MEUD.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 539
[16/40] Updating SMEA.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 576
[17/40] Updating XD9U.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 2651
[18/40] Updating XMME.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 2161
[19/40] Updating CSSX5E.MI...
    already up to date; no download needed
    downloaded 0 new rows; total rows now 4087
[20/40] Updating EMUL.MI...
    already up to date; no download needed
    downloaded 0 new rows

,ticker,status,downloaded_rows,total_rows,error
0,CSSPX.MI,ok,0,4038,
1,AVUV,ok,0,1645,
2,VTI,ok,0,4036,
3,VBR,ok,0,4024,
4,IEF,ok,0,4032,
5,VUTY.AS,ok,0,2592,
6,DBC,ok,0,4036,
7,GSG,ok,0,4036,
8,SWDA.MI,ok,0,4200,
9,EIMI.MI,ok,0,2935,
